# BESS Dispatch Optimiser (MILP)

Multi-site version of the day-ahead arbitrage optimizer. Choose the site and the time period at the top, then run all cells.

Model:
- Site offtake/injection (`off`/`inj`) are fixed, non-curtailable inputs the battery sits behind the meter with.
- Objective: minimize the cost of net grid exchange at the day-ahead price, plus a volumetric grid fee on offtake.
- Solved as a **MILP** (via OR-Tools' CBC backend) with binary variables enforcing that the battery never charges and discharges in the same interval, and the grid connection never draws and injects in the same interval -- made explicit with binaries rather than relying on the objective to discourage it.

## Data import -- choose site and period

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))
from srce.data_import import PROFILES
from srce.data_prep import select_first_n_days, select_date_range, select_representative_weeks

DATA_DIR = Path.cwd().parent / "data"
print("Known sites:", list(PROFILES))

Known sites: ['carmeuse', 'montea', 'lemahieu']


In [2]:
# ------------------------------ Choose site and period ------------------------------ #
SITE_NAME = "carmeuse"  # one of: "carmeuse", "montea", "lemahieu"
MODE = "weeks"          # "year", "days", "range", or "weeks"

N_DAYS = 5                     # used when MODE == "days"
START_DATE = "2024-01-01"      # used when MODE == "range" (inclusive)
END_DATE = "2024-01-06"        # used when MODE == "range" (exclusive)
DAYS_PER_MONTH = 7              # used when MODE == "weeks" -- first N days of each calendar month

if SITE_NAME not in PROFILES:
    raise ValueError(f"Unknown SITE_NAME {SITE_NAME!r}. Known sites: {sorted(PROFILES)}")

DATA_PATH = DATA_DIR / "csvs" / f"merged{SITE_NAME}ANDda_belgium.csv"

if MODE == "year":
    df = pd.read_csv(DATA_PATH, parse_dates=["dates"]).sort_values("dates")
elif MODE == "days":
    df = select_first_n_days(input_path=DATA_PATH, days=N_DAYS)
elif MODE == "range":
    df = select_date_range(input_path=DATA_PATH, start=START_DATE, end=END_DATE)
elif MODE == "weeks":
    # first `DAYS_PER_MONTH` days of each calendar month -- a compact,
    # seasonally-representative sample of the full year (12 x 7 days = 12
    # weeks by default), for optimizations that don't scale to a full year.
    df = select_representative_weeks(input_path=DATA_PATH, days_per_month=DAYS_PER_MONTH)
else:
    raise ValueError(f"Unknown MODE: {MODE!r}. Choose 'year', 'days', 'range', or 'weeks'.")

df["price"] = df["price [€/MWh]"]
print(f"{SITE_NAME}: {len(df)} rows, {df['dates'].min()} -> {df['dates'].max()}")
df.head()

carmeuse: 8064 rows, 2024-01-01 00:00:00+00:00 -> 2024-12-07 23:45:00+00:00


,dates,off,inj,gen,con,price [€/MWh],cleared_volume [MW],price
4,2024-01-01 00:00:00+00:00,748.0,0.0,0.0,748.0,0.01,518.00,0.01
5,2024-01-01 00:15:00+00:00,676.0,0.0,0.0,676.0,0.01,518.00,0.01
6,2024-01-01 00:30:00+00:00,648.0,0.0,0.0,648.0,0.01,518.00,0.01
7,2024-01-01 00:45:00+00:00,736.0,0.0,0.0,736.0,0.01,518.00,0.01
8,2024-01-01 01:00:00+00:00,720.0,0.0,0.0,720.0,0.00,560.65,0.00


## Packages

In [3]:
from ortools.linear_solver import pywraplp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [8]:
from srce.load_profile_analysis import analyze_load_profile
insight = analyze_load_profile(df, date_col="dates", cols=("con", "gen", "off", "inj"))
print(insight)

{'total_kwh': {'con': np.float64(8875167.6375), 'gen': np.float64(4996842.449621567), 'off': np.float64(4877550.103676244), 'inj': np.float64(999224.9157978096)}, 'peaks': {'con': {'value_kw': np.float64(2257.0875), 'timestamp': Timestamp('2024-03-06 14:30:00+0000', tz='UTC')}, 'gen': {'value_kw': np.float64(2905.7141401098897), 'timestamp': Timestamp('2024-05-01 12:00:00+0000', tz='UTC')}, 'off': {'value_kw': np.float64(1740.0), 'timestamp': Timestamp('2024-02-02 01:45:00+0000', tz='UTC')}, 'inj': {'value_kw': np.float64(2087.86924175824), 'timestamp': Timestamp('2024-08-06 11:15:00+0000', tz='UTC')}}, 'quarter_hour_profile':                con         gen         off       inj
dates                                               
00:00  1103.190476  299.479837  803.710639  0.000000
00:15  1093.095238  298.315944  794.779295  0.000000
00:30  1085.904762  297.129780  788.774982  0.000000
00:45  1088.333333  301.382324  786.951010  0.000000
01:00  1089.809524  301.224878  788.584646  0.0

## Assumptions

Note: these are placeholder values, not necessarily right for every site -- e.g. `carmeuse`'s real offtake alone regularly exceeds a 2 MW connection limit, which will make the solver report infeasible unless `GRID_OFFTAKE_LIMIT_MW` is raised to match its actual connection capacity. Check the site's typical load (e.g. via `srce.load_profile_analysis.analyze_load_profile`) before trusting a result.

In [ ]:
# ----------------------------- Assumptions ----------------------------- #
CAPACITY_MWH = 15.0      # battery energy capacity
INITIAL_SOC_MWH = 0.0   # battery starts empty
DT_HOURS = 0.25         # fixed 15-minute timestep
CHARGE_POWER_MW = 7.5     # max charging power
DISCHARGE_POWER_MW = 7.5  # max discharging power
CHARGE_EFFICIENCY = 0.95   # charging efficiency
DISCHARGE_EFFICIENCY = 0.95 # discharging efficiency
GRID_OFFTAKE_LIMIT_MW = 15    # max net power the site may draw from the grid
GRID_INJECTION_LIMIT_MW = 15  # max net power the site may feed into the grid
GRID_FEE_OFFTAKE_EUR_MWH = 54.205075   # volumetric grid fee charged per MWh drawn from the grid (placeholder)
GRID_FEE_INJECTION_EUR_MWH = 0.0  # volumetric grid fee charged per MWh fed into the grid (placeholder, often 0)
MAX_CYCLES_PER_DAY = 2   # cap on equivalent full charge cycles per calendar day (degradation limit)




#----------------------------- Derived Parameters ----------------------------- #
MAX_CHARGE_MWH = CHARGE_POWER_MW * DT_HOURS
MAX_DISCHARGE_MWH = DISCHARGE_POWER_MW * DT_HOURS
MAX_OFFTAKE_MWH = GRID_OFFTAKE_LIMIT_MW * DT_HOURS
MAX_INJECTION_MWH = GRID_INJECTION_LIMIT_MW * DT_HOURS

## Prepare the load data (kWh per interval -> MWh)

In [5]:
df["off_mwh"] = df["off"] / 1000.0   # kWh -> MWh for this interval
df["inj_mwh"] = df["inj"] / 1000.0
print(df.head())
n = len(df)

total_energy_mwh = df["off_mwh"].sum() - df["inj_mwh"].sum()
print(f"Net energy offtake over the period: {total_energy_mwh:.2f} MWh")
print(f"Max offtake (MWh/interval): {df['off_mwh'].max()}, Max injection (MWh/interval): {df['inj_mwh'].max()}")
print(f"Grid connection headroom per interval: offtake {MAX_OFFTAKE_MWH} MWh, injection {MAX_INJECTION_MWH} MWh")

                      dates    off  inj  gen    con  price [€/MWh]  \
4 2024-01-01 00:00:00+00:00  748.0  0.0  0.0  748.0           0.01   
5 2024-01-01 00:15:00+00:00  676.0  0.0  0.0  676.0           0.01   
6 2024-01-01 00:30:00+00:00  648.0  0.0  0.0  648.0           0.01   
7 2024-01-01 00:45:00+00:00  736.0  0.0  0.0  736.0           0.01   
8 2024-01-01 01:00:00+00:00  720.0  0.0  0.0  720.0           0.00   

   cleared_volume [MW]  price  off_mwh  inj_mwh  
4               518.00   0.01    0.748      0.0  
5               518.00   0.01    0.676      0.0  
6               518.00   0.01    0.648      0.0  
7               518.00   0.01    0.736      0.0  
8               560.65   0.00    0.720      0.0  
Net energy offtake over the period: 3878.33 MWh
Max offtake (MWh/interval): 1.74, Max injection (MWh/interval): 2.08786924175824
Grid connection headroom per interval: offtake 3.75 MWh, injection 3.75 MWh


## Input data check -- offtake, injection, and price as given

Sanity check the raw inputs before handing them to the solver.

In [39]:
fig_input = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.35, 0.35],
    vertical_spacing=0.08,
    subplot_titles=(
        f"{SITE_NAME.title()} day-ahead price (EUR/MWh)",
        "Site offtake / injection as given (MWh per interval)",
    ),
)

fig_input.add_trace(
    go.Scatter(x=df["dates"], y=df["price"], mode="lines", name="Price",
               line=dict(color="#2a78d6", width=2, shape="hv"),
               fill="tozeroy", fillcolor="rgba(42,120,214,0.08)"),
    row=1, col=1,
)

fig_input.add_trace(
    go.Scatter(x=df["dates"], y=df["off_mwh"], mode="lines", name="off",
               line=dict(color="#2a78d6", width=2, shape="hv"),
               fill="tozeroy", fillcolor="rgba(42,120,214,0.08)"),
    row=2, col=1,
)
fig_input.add_trace(
    go.Scatter(x=df["dates"], y=df["inj_mwh"], mode="lines", name="inj",
               line=dict(color="#e34948", width=2, shape="hv")),
    row=2, col=1,
)

fig_input.update_layout(
    height=700,
    showlegend=True,
    margin=dict(t=40, r=20, l=50, b=40),
    template="plotly_white",
    title=f"Input data as given -- {SITE_NAME}",
)
fig_input.update_yaxes(title_text="EUR/MWh", row=1, col=1)
fig_input.update_yaxes(title_text="MWh", row=2, col=1)

fig_input.show(renderer="browser")

## Build and solve the MILP

In [ ]:
# ------------------------------ Build the MILP ------------------------------ #
solver = pywraplp.Solver.CreateSolver("CBC")
if solver is None:
    raise RuntimeError("Could not create the CBC MILP solver -- check the OR-Tools install.")

# continuous decision variables: charge[t], discharge[t], soc[t] for each timestep t
charge = [solver.NumVar(0, MAX_CHARGE_MWH, f"charge_{t}") for t in range(n)]
discharge = [solver.NumVar(0, MAX_DISCHARGE_MWH, f"discharge_{t}") for t in range(n)]
soc = [solver.NumVar(0, CAPACITY_MWH, f"soc_{t}") for t in range(n)]
grid_offtake = [solver.NumVar(0, MAX_OFFTAKE_MWH, f"grid_offtake_{t}") for t in range(n)]
grid_injection = [solver.NumVar(0, MAX_INJECTION_MWH, f"grid_injection_{t}") for t in range(n)]

# binary decision variables -- this is what makes it a MILP: they force the
# solver to never charge+discharge, or draw+inject, in the same interval,
# instead of relying on the objective to make that unprofitable on its own.
is_charging = [solver.BoolVar(f"is_charging_{t}") for t in range(n)]
is_grid_offtake = [solver.BoolVar(f"is_grid_offtake_{t}") for t in range(n)]

for t in range(n):
    prev_soc = INITIAL_SOC_MWH if t == 0 else soc[t - 1]
    solver.Add(soc[t] == prev_soc + charge[t] * CHARGE_EFFICIENCY - discharge[t] / DISCHARGE_EFFICIENCY)

    # mutual exclusivity: charge or discharge, never both
    solver.Add(charge[t] <= MAX_CHARGE_MWH * is_charging[t])
    solver.Add(discharge[t] <= MAX_DISCHARGE_MWH * (1 - is_charging[t]))

    # net exchange at the grid connection point: load/generation (off, inj) have
    # priority -- they're fixed inputs the battery cannot change. The battery
    # can only adjust charge[t]/discharge[t] to keep the *combined* result
    # within the connection limits.
    net_grid_t = df["off_mwh"].iloc[t] - df["inj_mwh"].iloc[t] + charge[t] - discharge[t]
    solver.Add(grid_offtake[t] - grid_injection[t] == net_grid_t)

    # mutual exclusivity: draw from the grid or inject into it, never both
    solver.Add(grid_offtake[t] <= MAX_OFFTAKE_MWH * is_grid_offtake[t])
    solver.Add(grid_injection[t] <= MAX_INJECTION_MWH * (1 - is_grid_offtake[t]))

# cycle limit: an "equivalent full cycle" is CAPACITY_MWH worth of charging
# throughput. Capping total charge[t] per calendar day at
# MAX_CYCLES_PER_DAY * CAPACITY_MWH limits the battery to that many full
# charge cycles per day (a common degradation-management constraint), without
# needing extra binaries to count discrete charge/discharge events.
for day, day_idx in df.groupby(df["dates"].dt.date).indices.items():
    solver.Add(
        solver.Sum([charge[t] for t in day_idx]) <= MAX_CYCLES_PER_DAY * CAPACITY_MWH
    )

# cost[t] = price[t] * (off[t] - inj[t] + charge[t] - discharge[t]) + grid fees
# (off - inj) is a fixed number, not a variable, so it doesn't change where
# the optimum is -- it only affects the total cost we report afterwards
objective = solver.Objective()
for t in range(n):
    price = df["price"].iloc[t]
    objective.SetCoefficient(charge[t], price)
    objective.SetCoefficient(discharge[t], -price)
    objective.SetCoefficient(grid_offtake[t], GRID_FEE_OFFTAKE_EUR_MWH)
    objective.SetCoefficient(grid_injection[t], -GRID_FEE_INJECTION_EUR_MWH)
objective.SetMinimization()

status = solver.Solve()
if status != pywraplp.Solver.OPTIMAL:
    raise RuntimeError(
        "Solver did not find an optimal solution -- if you tightened the grid "
        "connection limits, check that off/inj alone (without the battery) "
        "never exceed them, since load/generation are fixed and cannot be curtailed. "
        f"(status code: {status})"
    )
print("Solved to optimality.")

## Results

In [ ]:
# -------------------------------- Results --------------------------------- #
df["charge_mwh"] = [v.solution_value() for v in charge]
df["discharge_mwh"] = [v.solution_value() for v in discharge]
df["soc_mwh"] = [v.solution_value() for v in soc]
df["grid_offtake_mwh"] = [v.solution_value() for v in grid_offtake]
df["grid_injection_mwh"] = [v.solution_value() for v in grid_injection]

df["net_grid_baseline"] = df["off_mwh"] - df["inj_mwh"]
df["net_grid_with_bess"] = df["grid_offtake_mwh"] - df["grid_injection_mwh"]

# split the post-BESS net flow back into offtake (>=0) and injection (>=0) components,
# so it can be plotted the same way as the original off/inj profiles
df["off_mwh_after_bess"] = df["net_grid_with_bess"].clip(lower=0)
df["inj_mwh_after_bess"] = (-df["net_grid_with_bess"]).clip(lower=0)

baseline_energy_cost = (df["net_grid_baseline"] * df["price"]).sum()
baseline_fee_cost = (df["off_mwh"] * GRID_FEE_OFFTAKE_EUR_MWH - df["inj_mwh"] * GRID_FEE_INJECTION_EUR_MWH).sum()
baseline_cost = baseline_energy_cost + baseline_fee_cost

bess_energy_cost = (df["net_grid_with_bess"] * df["price"]).sum()
bess_fee_cost = (df["grid_offtake_mwh"] * GRID_FEE_OFFTAKE_EUR_MWH - df["grid_injection_mwh"] * GRID_FEE_INJECTION_EUR_MWH).sum()
bess_cost = bess_energy_cost + bess_fee_cost

# "legs" that never cross the grid connection: the portion of the site's raw
# offtake/injection that the battery absorbs internally (load <-> battery,
# generation <-> battery) instead of it being metered as grid_offtake/
# grid_injection. Only the metered volumes ever pay grid fees, so this is
# exactly the no-BESS vs with-BESS volume difference. Can go negative if the
# battery is net *increasing* grid volume (e.g. buying low / selling high
# for pure arbitrage, beyond what's needed to serve the site itself).
offtake_leg_mwh = df["off_mwh"].sum() - df["grid_offtake_mwh"].sum()
injection_leg_mwh = df["inj_mwh"].sum() - df["grid_injection_mwh"].sum()

# self-consumption: share of on-site generation consumed on-site (directly or
# via the battery) rather than exported to the grid. Only meaningful if the
# site has "gen" data (not the case for e.g. montea, which only has off/inj).
if "gen" in df.columns and df["gen"].notna().any() and df["gen"].sum() > 0:
    gen_mwh_sum = df["gen"].sum() / 1000.0
    self_consumption_no_bess = 1 - (df["inj_mwh"].sum() / gen_mwh_sum)
    self_consumption_with_bess = 1 - (df["grid_injection_mwh"].sum() / gen_mwh_sum)
else:
    self_consumption_no_bess = self_consumption_with_bess = None

print(f"Site: {SITE_NAME}")
print(f"Period considered:         {df['dates'].min()} -> {df['dates'].max()}  ({len(df)} intervals)")
print(f"Average DA price:          € {df['price'].mean():,.2f} / MWh")
print()
print(f"Baseline cost (no BESS):   € {baseline_cost:,.2f}  (energy € {baseline_energy_cost:,.2f} + fees € {baseline_fee_cost:,.2f})")
print(f"Cost with BESS:            € {bess_cost:,.2f}  (energy € {bess_energy_cost:,.2f} + fees € {bess_fee_cost:,.2f})")
print(f"Savings from arbitrage:    € {baseline_cost - bess_cost:,.2f}")
print()
print(f"Offtake volume  -- no BESS: {df['off_mwh'].sum():,.2f} MWh   |   with BESS: {df['grid_offtake_mwh'].sum():,.2f} MWh")
print(f"Injection volume -- no BESS: {df['inj_mwh'].sum():,.2f} MWh   |   with BESS: {df['grid_injection_mwh'].sum():,.2f} MWh")
print()
print(f"Offtake leg volume  (site load <-> battery, never crossed the grid):       {offtake_leg_mwh:,.2f} MWh")
print(f"Injection leg volume (site generation <-> battery, never crossed the grid): {injection_leg_mwh:,.2f} MWh")
print()
if self_consumption_no_bess is not None:
    print(f"Self-consumption -- no BESS: {self_consumption_no_bess:.1%}   |   with BESS: {self_consumption_with_bess:.1%}")
else:
    print("Self-consumption: n/a (no on-site generation data for this site)")

## Plots

In [42]:
fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    row_heights=[0.35, 0.35, 0.3, 0.3, 0.3],
    vertical_spacing=0.06,
    subplot_titles=(
        f"{SITE_NAME.title()} day-ahead price (EUR/MWh)",
        "Battery state of charge (MWh)",
        "Charge / discharge (MWh per interval)",
        "Site offtake (MWh per interval)",
        "Site injection (MWh per interval)",
    ),
)

# --- price ---
fig.add_trace(
    go.Scatter(x=df["dates"], y=df["price"], mode="lines", name="Price",
               line=dict(color="#2a78d6", width=2, shape="hv"),
               fill="tozeroy", fillcolor="rgba(42,120,214,0.08)"),
    row=1, col=1,
)

# --- state of charge ---
fig.add_trace(
    go.Scatter(x=df["dates"], y=df["soc_mwh"], mode="lines", name="SoC",
               line=dict(color="#1baf7a", width=2, shape="hv"),
               fill="tozeroy", fillcolor="rgba(27,175,122,0.12)"),
    row=2, col=1,
)

# --- charge (positive) / discharge (negative, for visual contrast) ---
fig.add_trace(
    go.Bar(x=df["dates"], y=df["charge_mwh"], name="Charge", marker_color="#1baf7a"),
    row=3, col=1,
)
fig.add_trace(
    go.Bar(x=df["dates"], y=-df["discharge_mwh"], name="Discharge", marker_color="#e34948"),
    row=3, col=1,
)

# --- offtake ---
fig.add_trace(
    go.Scatter(x=df["dates"], y=df["off_mwh"], mode="lines", name="off",
               line=dict(color="#2a78d6", width=2, shape="hv"),
               fill="tozeroy", fillcolor="rgba(42,120,214,0.08)"),
    row=4, col=1,
)
fig.add_trace(
    go.Scatter(x=df["dates"], y=df["off_mwh_after_bess"], mode="lines", name="off (with BESS)",
               line=dict(color="#e39b1b", width=2, shape="hv")),
    row=4, col=1,
)

# --- injection ---
fig.add_trace(
    go.Scatter(x=df["dates"], y=df["inj_mwh"], mode="lines", name="inj",
               line=dict(color="#2a78d6", width=2, shape="hv"),
               fill="tozeroy", fillcolor="rgba(42,120,214,0.08)"),
    row=5, col=1,
)
fig.add_trace(
    go.Scatter(x=df["dates"], y=df["inj_mwh_after_bess"], mode="lines", name="inj (with BESS)",
               line=dict(color="#e39b1b", width=2, shape="hv")),
    row=5, col=1,
)

# --- capacity limit line on the SoC panel ---
fig.add_hline(
    y=CAPACITY_MWH, row=2, col=1,
    line=dict(color="#555", width=1, dash="dot"),
    annotation_text=f"Capacity ({CAPACITY_MWH} MWh)", annotation_position="top left",
)

# --- power limit lines on the charge/discharge panel ---
fig.add_hline(
    y=MAX_CHARGE_MWH, row=3, col=1,
    line=dict(color="#1baf7a", width=1, dash="dot"),
    annotation_text=f"Charge limit ({CHARGE_POWER_MW} MW)", annotation_position="top left",
)
fig.add_hline(
    y=-MAX_DISCHARGE_MWH, row=3, col=1,
    line=dict(color="#e34948", width=1, dash="dot"),
    annotation_text=f"Discharge limit ({DISCHARGE_POWER_MW} MW)", annotation_position="bottom left",
)

# --- power limit lines on the offtake and injection panels ---
fig.add_hline(
    y=MAX_OFFTAKE_MWH, row=4, col=1,
    line=dict(color="#1baf7a", width=1, dash="dot"),
    annotation_text=f"GC offtake limit ({MAX_OFFTAKE_MWH} MWh)", annotation_position="top left",
)
fig.add_hline(
    y=MAX_INJECTION_MWH, row=5, col=1,
    line=dict(color="#e34948", width=1, dash="dot"),
    annotation_text=f"GC injection limit ({MAX_INJECTION_MWH} MWh)", annotation_position="bottom left",
)

fig.update_layout(
    height=1200,
    showlegend=True,
    bargap=0,
    margin=dict(t=40, r=20, l=50, b=40),
    template="plotly_white",
    title=f"BESS dispatch -- {SITE_NAME}",
)
fig.update_yaxes(title_text="EUR/MWh", row=1, col=1)
fig.update_yaxes(title_text="MWh", row=2, col=1)
fig.update_yaxes(title_text="MWh", row=3, col=1)
fig.update_yaxes(title_text="MWh", row=4, col=1)
fig.update_yaxes(title_text="MWh", row=5, col=1)

fig.show(renderer="browser")